# 07 - RoBERTa Fine-Tuning for Misinformation Classification

This notebook fine-tunes `roberta-base` on each of the 3 datasets:
- **Manchester** - Rumour classification (reliable vs misinformation)
- **Monkeypox** - Misinformation detection
- **PHEME** - Rumour detection

### Strategy
- Full gold-standard data for final model training
- Stratified 5-Fold CV for robust evaluation
- Evaluation: F1 (macro), Precision, Recall, Accuracy, Confusion Matrix
- Hardware: Local CUDA GPU
- Results saved to `results/predictions/` and `results/models/`

## 0. Dataset Selection

**Change `DATASET` to switch between datasets.**

In [ ]:
# ============================================================
# DATASET SELECTION — change this to switch datasets
# Options: 'manchester', 'monkeypox', 'pheme'
# ============================================================
DATASET = 'manchester'

# Dataset configs
DATASET_CONFIG = {
    'manchester': {
        'train': '../../data/gold_standard/manchester_train.csv',
        'val':   '../../data/gold_standard/manchester_val.csv',
        'test':  '../../data/gold_standard/manchester_test.csv',
        'gold':  '../../data/gold_standard/manchester_gold_standard.csv',
        'text_col': 'cleaned_tweet',
        'label_col': 'label',
        'label_map': {'reliable': 0, 'misinformation': 1},
        'label_names': ['reliable', 'misinformation'],
        'pos_label': 'misinformation',
    },
    'monkeypox': {
        'train': '../../data/gold_standard/monkeypox_train.csv',
        'val':   '../../data/gold_standard/monkeypox_val.csv',
        'test':  '../../data/gold_standard/monkeypox_test.csv',
        'gold':  '../../data/gold_standard/monkeypox_gold_standard.csv',
        'text_col': 'cleaned_tweet',
        'label_col': 'label',
        'label_map': {'reliable': 0, 'misinformation': 1},
        'label_names': ['reliable', 'misinformation'],
        'pos_label': 'misinformation',
    },
    'pheme': {
        'train': '../../data/gold_standard/pheme_train.csv',
        'val':   '../../data/gold_standard/pheme_val.csv',
        'test':  '../../data/gold_standard/pheme_test.csv',
        'gold':  '../../data/gold_standard/pheme_gold_standard.csv',
        'text_col': 'cleaned_tweet',
        'label_col': 'label',
        'label_map': {'not_rumour': 0, 'rumour': 1},
        'label_names': ['not_rumour', 'rumour'],
        'pos_label': 'rumour',
    },
}

CFG = DATASET_CONFIG[DATASET]
print(f"Dataset: {DATASET.upper()}")
print(f"Text column: {CFG['text_col']}")
print(f"Label column: {CFG['label_col']}")
print(f"Labels: {CFG['label_names']}")

## 1. Imports & Setup

In [ ]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    RobertaTokenizerFast,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)
from datasets import Dataset as HFDataset

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

# Paths
RESULTS_DIR = Path('../../results')
MODELS_DIR  = RESULTS_DIR / 'models'
PREDS_DIR   = RESULTS_DIR / 'predictions'
FIGS_DIR    = RESULTS_DIR / 'figures'
for d in [MODELS_DIR, PREDS_DIR, FIGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# CUDA
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Load Data

In [ ]:
# Load splits
df_train = pd.read_csv(CFG['train'])
df_val   = pd.read_csv(CFG['val'])
df_test  = pd.read_csv(CFG['test'])
df_gold  = pd.read_csv(CFG['gold'])

TEXT_COL  = CFG['text_col']
LABEL_COL = CFG['label_col']
LABEL_MAP = CFG['label_map']
ID2LABEL  = {v: k for k, v in LABEL_MAP.items()}
NUM_LABELS = len(LABEL_MAP)

def drop_na_logged(df, subset, name):
    """Drop rows with NaN in subset columns, logging how many and why."""
    before = len(df)
    for col in subset:
        n_nan = df[col].isna().sum()
        if n_nan > 0:
            print(f"  [{name}] Dropping {n_nan:,} rows where '{col}' is NaN")
    df = df.dropna(subset=subset).copy()
    after = len(df)
    dropped = before - after
    if dropped == 0:
        print(f"  [{name}] No rows dropped (all {before:,} rows have valid text/label)")
    else:
        print(f"  [{name}] Dropped {dropped:,} rows total ({dropped/before*100:.1f}%) -> {after:,} remaining")
    df[TEXT_COL] = df[TEXT_COL].astype(str)
    return df

print("=== Dropped-Rows Audit ===")
df_train = drop_na_logged(df_train, [TEXT_COL, LABEL_COL], 'train')
df_val   = drop_na_logged(df_val,   [TEXT_COL, LABEL_COL], 'val')
df_test  = drop_na_logged(df_test,  [TEXT_COL, LABEL_COL], 'test')
df_gold  = drop_na_logged(df_gold,  [TEXT_COL, LABEL_COL], 'gold')

print(f"\nTrain: {len(df_train):,} | Val: {len(df_val):,} | Test: {len(df_test):,}")
print(f"\nTrain label distribution:")
print(df_train[LABEL_COL].value_counts())
print(f"\nTest label distribution:")
print(df_test[LABEL_COL].value_counts())

## 3. Tokenizer & Dataset Class

In [ ]:
MODEL_NAME = 'roberta-base'
MAX_LEN    = 128  # RoBERTa max, tweets are short

tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME)

class TweetDataset(Dataset):
    """PyTorch Dataset for tokenized tweets."""
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding='max_length',
            max_length=max_len,
            return_tensors='pt',
        )
        self.labels = torch.tensor(list(labels), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

def encode_labels(df):
    return df[LABEL_COL].map(LABEL_MAP).values

print(f"Tokenizer: {MODEL_NAME}")
print(f"Max sequence length: {MAX_LEN}")
print(f"Label map: {LABEL_MAP}")

## 4. Metrics Function

In [ ]:
def compute_metrics(eval_pred):
    """HuggingFace Trainer metric function."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy':  accuracy_score(labels, preds),
        'f1_macro':  f1_score(labels, preds, average='macro'),
        'f1_weighted': f1_score(labels, preds, average='weighted'),
        'precision': precision_score(labels, preds, average='macro', zero_division=0),
        'recall':    recall_score(labels, preds, average='macro', zero_division=0),
    }

def full_report(y_true, y_pred, label_names, dataset_name, fold=None):
    """Print classification report and return metrics dict."""
    fold_str = f" (Fold {fold})" if fold is not None else ""
    print(f"\n--- {dataset_name}{fold_str} ---")
    print(classification_report(y_true, y_pred, target_names=label_names, digits=4))
    return {
        'accuracy':  accuracy_score(y_true, y_pred),
        'f1_macro':  f1_score(y_true, y_pred, average='macro'),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted'),
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_macro':    recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_pos': f1_score(y_true, y_pred, pos_label=LABEL_MAP[CFG['pos_label']], zero_division=0),
    }

print("Metrics functions defined.")

In [ ]:
## 4b. Class Weights & Weighted Trainer
# Compute balanced class weights to address imbalance (Manchester 87/13%, PHEME 79/21%)

def compute_class_weights_for_dataset(df):
    """Compute sklearn balanced class weights and return as a float tensor."""
    labels_arr = df[LABEL_COL].map(LABEL_MAP).values
    classes    = np.array(sorted(LABEL_MAP.values()))
    weights    = compute_class_weight('balanced', classes=classes, y=labels_arr)
    print(f"  Class weights: { {ID2LABEL[c]: round(w, 4) for c, w in zip(classes, weights)} }")
    return torch.tensor(weights, dtype=torch.float)

# Compute on training portion of gold standard (will be recomputed per fold if needed)
print("=== Class Weight Computation ===")
CLASS_WEIGHTS = compute_class_weights_for_dataset(df_gold)
print(f"  Tensor: {CLASS_WEIGHTS}")

class WeightedTrainer(Trainer):
    """HuggingFace Trainer subclass that applies per-class loss weights."""

    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights.to(DEVICE)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits  = outputs.logits
        loss_fn = nn.CrossEntropyLoss(weight=self.class_weights)
        loss    = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

print("WeightedTrainer defined.")

## 5. Training Arguments

In [ ]:
def get_training_args(output_dir, num_epochs=4, batch_size=16):
    """Return TrainingArguments optimized for tweet classification."""
    return TrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        learning_rate=2e-5,
        weight_decay=0.01,
        warmup_ratio=0.1,
        lr_scheduler_type='linear',
        eval_strategy='epoch',           # renamed from evaluation_strategy in transformers>=4.46
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1_macro',
        greater_is_better=True,
        logging_steps=50,
        logging_dir=str(output_dir / 'logs'),
        seed=SEED,
        fp16=torch.cuda.is_available(),  # Mixed precision on GPU
        report_to='none',                # Disable W&B / wandb
        dataloader_num_workers=0,        # Windows compatibility
    )

print("Training config:")
print(f"  Epochs: 4 | Batch size: 16 | LR: 2e-5")
print(f"  fp16 mixed precision: {torch.cuda.is_available()}")
print(f"  Best model: max f1_macro")

print("Starting Stratified 5-Fold Cross-Validation on gold standard data...")
print(f"Total gold samples: {len(df_gold):,}")

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

all_texts  = df_gold[TEXT_COL].values
all_labels = encode_labels(df_gold)

# ── Pre-tokenize ALL texts once before the fold loop ──────────────────────────
# This avoids re-tokenizing the same texts N_FOLDS times (~30% time saving).
print(f"\nPre-tokenizing {len(all_texts):,} gold-standard texts (done once for all folds)...")
cached_encodings = tokenizer(
    list(all_texts),
    truncation=True,
    padding='max_length',
    max_length=MAX_LEN,
    return_tensors='pt',
)
print("Pre-tokenization complete.")

class CachedTweetDataset(Dataset):
    """Dataset that indexes into pre-computed token tensors — no per-call tokenization."""
    def __init__(self, encodings, labels, indices):
        self.encodings = encodings
        self.labels    = torch.tensor(labels, dtype=torch.long)
        self.indices   = indices  # subset indices into the cached tensors

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, pos):
        idx  = self.indices[pos]
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[pos]
        return item

cv_results = []  # List of metric dicts per fold
all_oof_preds  = np.zeros(len(df_gold), dtype=int)  # Out-of-fold predictions
all_oof_labels = all_labels.copy()

for fold, (train_idx, val_idx) in enumerate(skf.split(all_texts, all_labels), 1):
    print(f"\n{'='*50}")
    print(f" FOLD {fold}/{N_FOLDS}")
    print(f"{'='*50}")
    print(f" Train: {len(train_idx):,} | Val: {len(val_idx):,}")

    # Build datasets using cached encodings — no re-tokenization
    fold_train = CachedTweetDataset(cached_encodings, all_labels[train_idx], train_idx)
    fold_val   = CachedTweetDataset(cached_encodings, all_labels[val_idx],   val_idx)

    # Compute per-fold class weights from training split only
    fold_class_weights = compute_class_weight(
        'balanced',
        classes=np.array(sorted(LABEL_MAP.values())),
        y=all_labels[train_idx],
    )
    fold_weights_tensor = torch.tensor(fold_class_weights, dtype=torch.float)

    # Fresh model for each fold
    model = RobertaForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label=ID2LABEL,
        label2id=LABEL_MAP,
    )

    fold_output_dir = MODELS_DIR / f'{DATASET}_fold{fold}'
    args = get_training_args(fold_output_dir)

    # Use WeightedTrainer to handle class imbalance
    trainer = WeightedTrainer(
        class_weights=fold_weights_tensor,
        model=model,
        args=args,
        train_dataset=fold_train,
        eval_dataset=fold_val,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    trainer.train()

    # Predict on val fold
    preds_output = trainer.predict(fold_val)
    fold_preds   = np.argmax(preds_output.predictions, axis=-1)
    fold_labels  = all_labels[val_idx]

    # Store OOF predictions
    all_oof_preds[val_idx] = fold_preds

    # Metrics for this fold
    metrics = full_report(fold_labels, fold_preds, CFG['label_names'], DATASET.upper(), fold=fold)
    metrics['fold'] = fold
    cv_results.append(metrics)

    # Free GPU memory
    del model, trainer, fold_train, fold_val
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nCross-validation complete!")

In [ ]:
print("Starting Stratified 5-Fold Cross-Validation on gold standard data...")
print(f"Total gold samples: {len(df_gold):,}")

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

all_texts  = df_gold[TEXT_COL].values
all_labels = encode_labels(df_gold)

cv_results = []  # List of metric dicts per fold
all_oof_preds  = np.zeros(len(df_gold), dtype=int)  # Out-of-fold predictions
all_oof_labels = all_labels.copy()

for fold, (train_idx, val_idx) in enumerate(skf.split(all_texts, all_labels), 1):
    print(f"\n{'='*50}")
    print(f" FOLD {fold}/{N_FOLDS}")
    print(f"{'='*50}")
    print(f" Train: {len(train_idx):,} | Val: {len(val_idx):,}")

    # Build datasets
    fold_train = TweetDataset(all_texts[train_idx], all_labels[train_idx], tokenizer)
    fold_val   = TweetDataset(all_texts[val_idx],   all_labels[val_idx],   tokenizer)

    # Fresh model for each fold
    model = RobertaForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label=ID2LABEL,
        label2id=LABEL_MAP,
    )

    fold_output_dir = MODELS_DIR / f'{DATASET}_fold{fold}'
    args = get_training_args(fold_output_dir)

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=fold_train,
        eval_dataset=fold_val,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    trainer.train()

    # Predict on val fold
    preds_output = trainer.predict(fold_val)
    fold_preds   = np.argmax(preds_output.predictions, axis=-1)
    fold_labels  = all_labels[val_idx]

    # Store OOF predictions
    all_oof_preds[val_idx] = fold_preds

    # Metrics for this fold
    metrics = full_report(fold_labels, fold_preds, CFG['label_names'], DATASET.upper(), fold=fold)
    metrics['fold'] = fold
    cv_results.append(metrics)

    # Free GPU memory
    del model, trainer, fold_train, fold_val
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nCross-validation complete!")

## 7. CV Results Summary

In [ ]:
## 7b. Per-Fold Metrics Heatmap
# Visualises how precision / recall / F1 vary across the 5 folds.

heatmap_metrics = ['accuracy', 'f1_macro', 'f1_weighted', 'precision_macro', 'recall_macro', 'f1_pos']
heatmap_labels  = ['Accuracy', 'F1 Macro', 'F1 Weighted', 'Precision', 'Recall', f'F1 {CFG["pos_label"]}']

heatmap_data = cv_df[heatmap_metrics].copy()
heatmap_data.columns = heatmap_labels

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(
    heatmap_data.T,
    annot=True,
    fmt='.4f',
    cmap='YlGnBu',
    vmin=0.5,
    vmax=1.0,
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': 'Score'},
)
ax.set_title(f'{DATASET.upper()} — Per-Fold Metrics Heatmap (5-Fold CV)', fontweight='bold', pad=12)
ax.set_xlabel('Fold')
ax.set_ylabel('Metric')
ax.set_xticklabels([f'Fold {i}' for i in cv_df.index])

plt.tight_layout()
heatmap_path = FIGS_DIR / f'{DATASET}_roberta_cv_heatmap.png'
plt.savefig(heatmap_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Heatmap saved: {heatmap_path}")

In [ ]:
cv_df = pd.DataFrame(cv_results).set_index('fold')

print(f"\n{'='*60}")
print(f" {DATASET.upper()} - 5-Fold CV Summary")
print(f"{'='*60}")
print(cv_df.round(4).to_string())

print(f"\n--- Mean +/- Std ---")
mean_std = cv_df.agg(['mean', 'std'])
print(mean_std.round(4).to_string())

# Overall OOF metrics
print(f"\n--- Overall Out-of-Fold (OOF) Metrics ---")
oof_metrics = full_report(all_oof_labels, all_oof_preds, CFG['label_names'], f"{DATASET.upper()} OOF")

# Save CV results
cv_df.to_csv(PREDS_DIR / f'{DATASET}_cv_results.csv')
print(f"\nCV results saved to results/predictions/{DATASET}_cv_results.csv")

## 8. CV Confusion Matrix (OOF)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- OOF Confusion Matrix ---
cm = confusion_matrix(all_oof_labels, all_oof_preds)
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CFG['label_names'],
    yticklabels=CFG['label_names'],
    ax=axes[0]
)
axes[0].set_title(f'{DATASET.upper()} - OOF Confusion Matrix (5-Fold CV)', fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

# --- Per-Fold F1 Bar Chart ---
folds = cv_df.index.tolist()
f1_scores = cv_df['f1_macro'].tolist()
colors = ['steelblue'] * N_FOLDS

bars = axes[1].bar(folds, f1_scores, color=colors, edgecolor='black', linewidth=0.7)
axes[1].axhline(np.mean(f1_scores), color='red', linestyle='--', linewidth=1.5, label=f'Mean: {np.mean(f1_scores):.4f}')
axes[1].set_xlabel('Fold')
axes[1].set_ylabel('F1 Macro')
axes[1].set_title(f'{DATASET.upper()} - F1 Macro per Fold', fontweight='bold')
axes[1].set_ylim([max(0, min(f1_scores) - 0.05), 1.0])
axes[1].legend()
for bar, val in zip(bars, f1_scores):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
fig_path = FIGS_DIR / f'{DATASET}_roberta_cv_results.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure saved: {fig_path}")

## 9. Final Model - Train on Full Train+Val, Evaluate on Test

In [ ]:
# Predict on test set
test_output  = final_trainer.predict(final_test_ds)
test_preds   = np.argmax(test_output.predictions, axis=-1)
test_probs   = torch.softmax(torch.tensor(test_output.predictions), dim=-1).numpy()

# Positive-class probabilities (used for ROC / PR curves)
pos_label_int  = LABEL_MAP[CFG['pos_label']]
test_probs_pos = test_probs[:, pos_label_int]

# Full classification report
test_metrics = full_report(test_labels, test_preds, CFG['label_names'], f"{DATASET.upper()} TEST")

# ROC-AUC and PR-AUC
roc_auc = roc_auc_score(test_labels, test_probs_pos)
pr_auc  = average_precision_score(test_labels, test_probs_pos)
test_metrics['roc_auc'] = roc_auc
test_metrics['pr_auc']  = pr_auc

print(f"\n--- Final Test Metrics ---")
for k, v in test_metrics.items():
    print(f"  {k:<25}: {v:.4f}")

In [ ]:
## 10b. ROC Curve and Precision-Recall Curve

fpr, tpr, _    = roc_curve(test_labels, test_probs_pos)
prec_c, rec_c, _ = precision_recall_curve(test_labels, test_probs_pos)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- ROC Curve ---
axes[0].plot(fpr, tpr, color='steelblue', lw=2, label=f'ROC AUC = {roc_auc:.4f}')
axes[0].plot([0, 1], [0, 1], color='grey', linestyle='--', lw=1, label='Random')
axes[0].set_xlim([0.0, 1.0])
axes[0].set_ylim([0.0, 1.05])
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title(f'{DATASET.upper()} — ROC Curve (Test Set)', fontweight='bold')
axes[0].legend(loc='lower right')
axes[0].grid(alpha=0.3)

# --- Precision-Recall Curve ---
axes[1].plot(rec_c, prec_c, color='darkorange', lw=2, label=f'PR AUC = {pr_auc:.4f}')
# Baseline (prevalence)
baseline = test_labels.sum() / len(test_labels)
axes[1].axhline(baseline, color='grey', linestyle='--', lw=1, label=f'Baseline = {baseline:.3f}')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title(f'{DATASET.upper()} — Precision-Recall Curve (Test Set)', fontweight='bold')
axes[1].legend(loc='upper right')
axes[1].grid(alpha=0.3)

plt.tight_layout()
curve_path = FIGS_DIR / f'{DATASET}_roberta_roc_pr_curves.png'
plt.savefig(curve_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"ROC/PR curves saved: {curve_path}")
print(f"ROC-AUC : {roc_auc:.4f}")
print(f"PR-AUC  : {pr_auc:.4f}")

## 10. Final Test Set Evaluation

In [ ]:
# Predict on test set
test_output  = final_trainer.predict(final_test_ds)
test_preds   = np.argmax(test_output.predictions, axis=-1)
test_probs   = torch.softmax(torch.tensor(test_output.predictions), dim=-1).numpy()

# Full report
test_metrics = full_report(test_labels, test_preds, CFG['label_names'], f"{DATASET.upper()} TEST")

print(f"\n--- Final Test Metrics ---")
for k, v in test_metrics.items():
    print(f"  {k:<25}: {v:.4f}")

# Save test predictions with probabilities
df_test_preds = df_test[[TEXT_COL, LABEL_COL]].copy()
df_test_preds['true_label_int']  = test_labels
df_test_preds['pred_label_int']  = test_preds
df_test_preds['pred_label']      = [ID2LABEL[p] for p in test_preds]
for i, name in enumerate(CFG['label_names']):
    df_test_preds[f'prob_{name}'] = test_probs[:, i]
df_test_preds['correct'] = (df_test_preds['true_label_int'] == df_test_preds['pred_label_int'])

pred_path = PREDS_DIR / f'{DATASET}_roberta_test_predictions.csv'
df_test_preds.to_csv(pred_path, index=False)
print(f"Predictions saved: {pred_path}")

# Save final model
model_save_path = MODELS_DIR / f'{DATASET}_roberta_final'
final_trainer.save_model(str(model_save_path))
tokenizer.save_pretrained(str(model_save_path))
print(f"Model saved: {model_save_path}")

# Save complete results summary (includes ROC-AUC + PR-AUC)
summary = {
    'dataset': DATASET,
    'model': MODEL_NAME,
    'cv_f1_macro_mean': cv_df['f1_macro'].mean(),
    'cv_f1_macro_std':  cv_df['f1_macro'].std(),
    'test_accuracy':    test_metrics['accuracy'],
    'test_f1_macro':    test_metrics['f1_macro'],
    'test_f1_weighted': test_metrics['f1_weighted'],
    'test_precision':   test_metrics['precision_macro'],
    'test_recall':      test_metrics['recall_macro'],
    f'test_f1_{CFG["pos_label"]}': test_metrics['f1_pos'],
    'test_roc_auc':     test_metrics.get('roc_auc', float('nan')),
    'test_pr_auc':      test_metrics.get('pr_auc',  float('nan')),
}

summary_path = PREDS_DIR / f'{DATASET}_roberta_summary.csv'
pd.DataFrame([summary]).to_csv(summary_path, index=False)
print(f"Summary saved: {summary_path}")

print("\n=== FINAL RESULTS ===")
for k, v in summary.items():
    if isinstance(v, float):
        print(f"  {k:<35}: {v:.4f}")
    else:
        print(f"  {k:<35}: {v}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Confusion Matrix ---
cm_test = confusion_matrix(test_labels, test_preds)
cm_norm = cm_test.astype(float) / cm_test.sum(axis=1, keepdims=True)  # Row-normalized

# Raw counts
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues',
            xticklabels=CFG['label_names'],
            yticklabels=CFG['label_names'], ax=axes[0])
axes[0].set_title(f'{DATASET.upper()} - Test Confusion Matrix (Counts)', fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

# Normalized
sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=CFG['label_names'],
            yticklabels=CFG['label_names'], ax=axes[1])
axes[1].set_title(f'{DATASET.upper()} - Test Confusion Matrix (Normalized)', fontweight='bold')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')

plt.tight_layout()
fig_path = FIGS_DIR / f'{DATASET}_roberta_test_cm.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure saved: {fig_path}")

## 12. Save Predictions & Model

In [ ]:
# Save test predictions with probabilities
df_test_preds = df_test[[TEXT_COL, LABEL_COL]].copy()
df_test_preds['true_label_int']  = test_labels
df_test_preds['pred_label_int']  = test_preds
df_test_preds['pred_label']      = [ID2LABEL[p] for p in test_preds]
for i, name in enumerate(CFG['label_names']):
    df_test_preds[f'prob_{name}'] = test_probs[:, i]
df_test_preds['correct'] = (df_test_preds['true_label_int'] == df_test_preds['pred_label_int'])

pred_path = PREDS_DIR / f'{DATASET}_roberta_test_predictions.csv'
df_test_preds.to_csv(pred_path, index=False)
print(f"Predictions saved: {pred_path}")

# Save final model
model_save_path = MODELS_DIR / f'{DATASET}_roberta_final'
final_trainer.save_model(str(model_save_path))
tokenizer.save_pretrained(str(model_save_path))
print(f"Model saved: {model_save_path}")

# Save complete results summary
summary = {
    'dataset': DATASET,
    'model': MODEL_NAME,
    'cv_f1_macro_mean': cv_df['f1_macro'].mean(),
    'cv_f1_macro_std':  cv_df['f1_macro'].std(),
    'test_accuracy':    test_metrics['accuracy'],
    'test_f1_macro':    test_metrics['f1_macro'],
    'test_f1_weighted': test_metrics['f1_weighted'],
    'test_precision':   test_metrics['precision_macro'],
    'test_recall':      test_metrics['recall_macro'],
    f'test_f1_{CFG["pos_label"]}': test_metrics['f1_pos'],
}

summary_path = PREDS_DIR / f'{DATASET}_roberta_summary.csv'
pd.DataFrame([summary]).to_csv(summary_path, index=False)
print(f"Summary saved: {summary_path}")

print("\n=== FINAL RESULTS ===")
for k, v in summary.items():
    if isinstance(v, float):
        print(f"  {k:<35}: {v:.4f}")
    else:
        print(f"  {k:<35}: {v}")

## 13. Error Analysis - Misclassified Samples

In [ ]:
errors = df_test_preds[~df_test_preds['correct']].copy()
errors['true_label'] = errors[LABEL_COL]

print(f"Total misclassified: {len(errors):,} / {len(df_test_preds):,} ({len(errors)/len(df_test_preds)*100:.1f}%)")
print(f"\nError breakdown by true label:")
print(errors['true_label'].value_counts())

# False Positives (predicted misinformation, actually reliable)
pos_label_int = LABEL_MAP[CFG['pos_label']]
fp = errors[(errors['pred_label_int'] == pos_label_int) & (errors['true_label_int'] != pos_label_int)]
fn = errors[(errors['pred_label_int'] != pos_label_int) & (errors['true_label_int'] == pos_label_int)]

print(f"\nFalse Positives (predicted {CFG['pos_label']}, actually not): {len(fp)}")
print(f"False Negatives (missed {CFG['pos_label']}): {len(fn)}")

print(f"\n--- Sample False Negatives (missed {CFG['pos_label']}) ---")
if len(fn) > 0:
    for _, row in fn.head(5).iterrows():
        prob_col = f'prob_{CFG["pos_label"]}'
        print(f"  [{row[prob_col]:.3f}] {row[TEXT_COL][:120]}")

print(f"\n--- Sample False Positives ---")
if len(fp) > 0:
    for _, row in fp.head(5).iterrows():
        prob_col = f'prob_{CFG["pos_label"]}'
        print(f"  [{row[prob_col]:.3f}] {row[TEXT_COL][:120]}")

## 14. Metrics Summary Table

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('off')

metrics_display = [
    ['Metric', 'CV Mean (5-Fold)', 'CV Std', 'Test Set'],
    ['F1 Macro',
     f"{cv_df['f1_macro'].mean():.4f}",
     f"+/-{cv_df['f1_macro'].std():.4f}",
     f"{test_metrics['f1_macro']:.4f}"],
    ['F1 Weighted',
     f"{cv_df['f1_weighted'].mean():.4f}",
     f"+/-{cv_df['f1_weighted'].std():.4f}",
     f"{test_metrics['f1_weighted']:.4f}"],
    ['Accuracy',
     f"{cv_df['accuracy'].mean():.4f}",
     f"+/-{cv_df['accuracy'].std():.4f}",
     f"{test_metrics['accuracy']:.4f}"],
    ['Precision (Macro)',
     f"{cv_df['precision_macro'].mean():.4f}",
     f"+/-{cv_df['precision_macro'].std():.4f}",
     f"{test_metrics['precision_macro']:.4f}"],
    ['Recall (Macro)',
     f"{cv_df['recall_macro'].mean():.4f}",
     f"+/-{cv_df['recall_macro'].std():.4f}",
     f"{test_metrics['recall_macro']:.4f}"],
    [f'F1 ({CFG["pos_label"]})', '-', '-', f"{test_metrics['f1_pos']:.4f}"],
]

table = ax.table(
    cellText=metrics_display[1:],
    colLabels=metrics_display[0],
    cellLoc='center',
    loc='center',
    bbox=[0, 0, 1, 1]
)
table.auto_set_font_size(False)
table.set_fontsize(11)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor('#2c3e50')
        cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#ecf0f1')

ax.set_title(f'{DATASET.upper()} - RoBERTa Results Summary', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
fig_path = FIGS_DIR / f'{DATASET}_roberta_summary_table.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Table saved: {fig_path}")